# Budget choice under a known acquisition
[Proof](../13_acquisition_only_budget_choice.md). The candidate family and prior are fixed before observing test events.

In [ ]:
import numpy as np
from experiments.sampled_conditioning.core import *
from experiments.sampled_conditioning.run import evaluate_design
t,A=sampled_design(1024,.2,30.)
J=A.T@A/.25
costs=[expected_plugin_kl(J,retained_matrix(J,k)) for k in range(3)]
selected=choose_pattern(J)
assert costs[selected]==min(costs)
header=encode_header(np.arange(4.),J,selected)
score,decoded=decode_header(header)
assert len(header)==11
np.testing.assert_allclose(decoded,retained_matrix(J,selected))
print('expected KL by candidate',costs,'selected',selected)

## Independent Monte Carlo checks the expectation
Same event and noisy waveform for every arm; no hidden coefficient enters the selector.

In [ ]:
rng=np.random.default_rng(44)
beta=rng.normal(size=(40000,4))
noise=rng.normal(size=(40000,len(A)))
result=evaluate_design(A,beta,noise,'within')
se=result['kl'].std(ddof=1)/np.sqrt(len(beta))
assert abs(result['kl'].mean()-result['expected_kl'])<5*se
print('Monte Carlo / design expectation / standard error',result['kl'].mean(),result['expected_kl'],se)
for arm in ['diag','within','magnitude','risk_oracle','full']:
    control=evaluate_design(A,beta[:64],noise[:64],arm,True)
    np.testing.assert_allclose(control['kl'],0.,atol=1e-10)
print('full-side-input control passed')

## Strong same-storage moment control
Exact encoder-side moments are an analytic control, not a learned tokenizer.

In [ ]:
plugin=evaluate_design(A,beta[:256],noise[:256],'within')
moment=evaluate_design(A,beta[:256],noise[:256],'moment_within')
assert np.all(moment['kl']<=plugin['kl']+1e-9)
np.testing.assert_allclose(moment['kl'],moment['expected_kl'],atol=1e-10)
print('mean KL: precision / moment',plugin['kl'].mean(),moment['kl'].mean())
print('THEORY_DEMO_PASS::13_acquisition_only_budget_choice')